In [ ]:
import kagglehub
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import shutil
import cv2
import numpy as np
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.optim as optim


In [ ]:
pathCF = kagglehub.dataset_download("tuyenldvn/caucafall")
pathLF = kagglehub.dataset_download("tuyenldvn/falldataset-imvia")

print("Path to dataset files:", pathCF)
print("Path to dataset files:", pathLF)


Using Colab cache for faster access to the 'caucafall' dataset.


100%|██████████| 9.37G/9.37G [01:32<00:00, 109MB/s] 

Extracting files...


Path to dataset files: /kaggle/input/caucafall
Path to dataset files: /root/.cache/kagglehub/datasets/tuyenldvn/falldataset-imvia/versions/2


In [ ]:
#-------------------------------------------------------------------------------
# Copying datasets into content/ from session's cache/
#-------------------------------------------------------------------------------

# Source directories
CFALL_SRC = "/root/.cache/kagglehub/datasets/tuyenldvn/caucafall/versions/2/Dataset CAUCAFall/CAUCAFall"
LFALL_SRC = "/root/.cache/kagglehub/datasets/tuyenldvn/falldataset-imvia/versions/2"
# Target directories
CFALL_DST = "/content/CAUCA_FALL"
LFALL_DST = "/content/LE2i_FALL"

def move_dataset(src, dst):
    if os.path.exists(dst):
        print(f"{dst} already exists. Skipping copy.")
    else:
        shutil.copytree(src, dst)
        print(f"Copied dataset to {dst}")

# Move datasets
move_dataset(CFALL_SRC, CFALL_DST)
move_dataset(LFALL_SRC, LFALL_DST)


Copied dataset to /content/CAUCA_FALL
Copied dataset to /content/LE2i_FALL


In [ ]:
# =========================
# Stage-1 Global Config
# =========================

# ---- Dataset roots ----
CFALL_DST = "/content/CAUCA_FALL"   # CAUCAFall (fall + non-fall)
LFALL_DST = "/content/LE2i_FALL"    # LE2I (fall-only)

# ---- Output root for Stage-1 ----
STAGE1_OUT = "/content/STAGE1_DATA"

# ---- Frame / clip parameters (MUST match Stage-2) ----
CLIP_LEN = 32          # number of frames per clip
FRAME_STRIDE = 1       # sampling stride (confirm = Stage-2)
FRAME_SIZE = (224,224) # resize (confirm = Stage-2)

# ---- Train / test protocol (subject-based) ----
TRAIN_SUBJECTS = {f"S{i}" for i in range(1, 9)}   # S1–S8
TEST_SUBJECTS  = {f"S{i}" for i in range(9, 11)}  # S9–S10

# ---- Sanity prints ----
print("CAUCAFall root:", CFALL_DST)
print("LE2I root     :", LFALL_DST)
print("Stage-1 out   :", STAGE1_OUT)
print("Clip length   :", CLIP_LEN)


CAUCAFall root: /content/CAUCA_FALL
LE2I root     : /content/LE2i_FALL
Stage-1 out   : /content/STAGE1_DATA
Clip length   : 32


In [ ]:
class Stage1_3DCNN(nn.Module):
    """
    Stage-1 spatio-temporal classifier.
    - Trained as fall / no-fall classifier
    - Can later output per-frame saliency weights
    """

    def __init__(self, num_classes=2):
        super().__init__()

        # -------- Spatio-temporal backbone --------
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3,7,7),
                               stride=(1,2,2), padding=(1,3,3))
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(kernel_size=(1,3,3),
                                  stride=(1,2,2), padding=(0,1,1))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(2,2,2),
                                  stride=(2,2,2))

        self.conv3 = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(256)

        self.conv4 = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm3d(256)


        # -------- Classification head --------
        self.global_pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.fc = nn.Linear(256, num_classes)

        self.dropout = nn.Dropout(p=0.5)


    def forward(self, x, return_saliency=False):
        """
        x: (B, 3, T, H, W)
        return_saliency:
            False -> return logits
            True  -> return (logits, saliency_weights)
        """

        # ---- Backbone ----
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))   # <-- SALIENCY LAYER

         # x shape: (B, C, T', H', W')

        # ---- Saliency computation (temporal importance) ----
        # Temporal saliency from intermediate features
        # Collapse channel + spatial dims, keep time
        saliency = x.detach().pow(2).mean(dim=(1, 3, 4))   # (B, T')

        # Normalize per clip (important!)
        saliency = saliency / (saliency.sum(dim=1, keepdim=True) + 1e-6)


        x = F.relu(self.bn4(self.conv4(x)))
        # ---- Classification head ----
        pooled = self.global_pool(x).flatten(1)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)

        if return_saliency:
            return logits, saliency
        else:
            return logits


In [ ]:
def extract_frames_from_video(
    video_path,
    clip_len=CLIP_LEN,
    frame_stride=FRAME_STRIDE,
    resize=FRAME_SIZE
):
    """
    Extracts frames from a video in a deterministic way.
    Output frames align EXACTLY with Stage-2 pose extraction.

    Returns:
        frames: list of RGB numpy arrays (len = clip_len)
    """

    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"Cannot open {video_path}"

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    assert total_frames > 0, "Empty video"

    # ---- uniform indices across entire video ----
    if total_frames >= CLIP_LEN:
        frame_indices = np.linspace(
            0, total_frames - 1, CLIP_LEN, dtype=int
        )
    else:
        frame_indices = list(range(total_frames))
        frame_indices += [total_frames - 1] * (CLIP_LEN - total_frames)
        frame_indices = np.array(frame_indices, dtype=int)

    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError(f"Failed to read frame {idx}")

        frame = cv2.resize(frame, (224,224))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()


    if len(frames) < clip_len:
        raise ValueError(f"Video too short: {video_path}")
    return frames

def build_clip_tensor(frames):
    """
    Converts list of RGB frames to Stage-1 input tensor.

    Input:
        frames: list of (H,W,3) RGB arrays

    Output:
        clip tensor: (1, 3, T, H, W)
    """

    frames = np.array(frames)                 # (T, H, W, 3)
    frames = frames.astype(np.float32) / 255.0

    clip = torch.from_numpy(frames)
    clip = clip.permute(3, 0, 1, 2)            # (3, T, H, W)
    return clip


In [ ]:
def build_caucafall_index(root):
    """
    Builds an index for CAUCAFall dataset.

    Returns:
        List of dicts with:
            - video_path
            - label (0 = no-fall, 1 = fall)
            - subject
            - dataset = 'caucafall'
    """
    index = []

    for subject_dir in sorted(Path(root).glob("Subject.*")):
        subject = subject_dir.name.replace("Subject.", "S")

        for activity_dir in subject_dir.iterdir():
            if not activity_dir.is_dir():
                continue

            activity_name = activity_dir.name.lower()

            is_fall = "fall" in activity_name
            label = 1 if is_fall else 0

            for video_file in activity_dir.glob("*.avi"):
                index.append({
                    "video_path": str(video_file),
                    "label": label,
                    "subject": subject,
                    "dataset": "caucafall"
                })

    return index

def build_le2i_index(root):
    """
    Builds an index for LE2I dataset (fall-only).

    Returns:
        List of dicts with:
            - video_path
            - label = 1 (fall)
            - subject = 'NA'
            - dataset = 'le2i'
    """
    index = []

    for video_file in Path(root).rglob("*.avi"):
        index.append({
            "video_path": str(video_file),
            "label": 1,
            "subject": "NA",
            "dataset": "le2i"
        })

    return index

caucafall_index = build_caucafall_index(CFALL_DST)
le2i_index = build_le2i_index(LFALL_DST)

full_index = caucafall_index

print("CAUCAFall videos:", len(caucafall_index))
#print("LE2I videos     :", len(le2i_index))
#print("Total videos    :", len(full_index))

print("Dataset split:", Counter(d["dataset"] for d in full_index))
print("Label split  :", Counter(d["label"] for d in full_index))


CAUCAFall videos: 100
Dataset split: Counter({'caucafall': 100})
Label split  : Counter({0: 50, 1: 50})


In [ ]:
class Stage1VideoDataset(Dataset):
    """
    Generic Stage-1 dataset for both CAUCAFall and LE2I.
    """

    def __init__(self, index):
        """
        index: list of dicts from Block 5
        """
        self.index = index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        entry = self.index[idx]

        video_path = entry["video_path"]
        label = entry["label"]

        # ---- Load clip (Stage-2 aligned) ----
        frames = extract_frames_from_video(video_path)
        clip = build_clip_tensor(frames)

        return clip, label


In [ ]:
def split_caucafall_by_subject(index, train_subjects, test_subjects):
    train_idx, test_idx = [], []

    for entry in index:
        if entry["dataset"] != "caucafall":
            continue

        if entry["subject"] in train_subjects:
            train_idx.append(entry)
        elif entry["subject"] in test_subjects:
            test_idx.append(entry)

    return train_idx, test_idx

# CAUCAFall splits
caucafall_train_idx, caucafall_test_idx = split_caucafall_by_subject(
    caucafall_index,
    TRAIN_SUBJECTS,
    TEST_SUBJECTS
)

# LE2I (fall-only, used only for pretraining)
#le2i_train_idx = le2i_index

print("CAUCAFall train:", len(caucafall_train_idx))
print("CAUCAFall test :", len(caucafall_test_idx))
#print("LE2I train     :", len(le2i_train_idx))
caucafall_train_ds = Stage1VideoDataset(caucafall_train_idx)
caucafall_test_ds  = Stage1VideoDataset(caucafall_test_idx)
#le2i_train_ds      = Stage1VideoDataset(le2i_train_idx)
BATCH_SIZE = 1   # video-level batches (safe for 3D CNNs)

caucafall_train_loader = DataLoader(
    caucafall_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

caucafall_test_loader = DataLoader(
    caucafall_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# le2i_train_loader = DataLoader(
#     le2i_train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,
#     pin_memory=True
# )


CAUCAFall train: 80
CAUCAFall test : 20


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Stage1_3DCNN(num_classes=2)
model.eval()
model = model.to(device)

# ---- Loss ----
pos_weight = torch.tensor([2.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(device))


# ---- Optimizer ----
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-5
)

# ---- Optional scheduler (safe default) ----
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for clips, labels in loader:
        clips = clips.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(clips)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total if total > 0 else 0.0

    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for clips, labels in loader:
        clips = clips.to(device)
        labels = labels.to(device)

        outputs = model(clips)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total if total > 0 else 0.0
    return acc

In [ ]:
# model = model.to(device)
# torch.cuda.empty_cache()

# PRETRAIN_EPOCHS = 5

# print("\n🔹 Phase 1: LE2I pretraining")

# for epoch in range(PRETRAIN_EPOCHS):
#     loss, acc = train_one_epoch(
#         model,
#         le2i_train_loader,
#         optimizer,
#         criterion,
#         device
#     )
#     scheduler.step()

#     print(f"[LE2I] Epoch {epoch+1}/{PRETRAIN_EPOCHS} | Loss: {loss:.4f}")


In [ ]:
clips, labels = next(iter(caucafall_train_loader))
clips = clips.to(device)

logits, sal = model(clips, return_saliency=True)
print(logits)
print(sal)


tensor([[ 0.0259, -0.0405]], grad_fn=<AddmmBackward0>)
tensor([[0.0415, 0.0651, 0.0667, 0.0665, 0.0660, 0.0655, 0.0648, 0.0647, 0.0648,
         0.0650, 0.0655, 0.0661, 0.0669, 0.0674, 0.0624, 0.0411]])


In [ ]:
model = Stage1_3DCNN(num_classes=2)
model = model.to(device)
model.train()

FINETUNE_EPOCHS = 10

print("\n🔹 Phase 2: CAUCAFall fine-tuning")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        caucafall_train_loader,
        optimizer,
        criterion,
        device
    )

    test_acc = evaluate(
        model,
        caucafall_test_loader,
        device
    )

    scheduler.step()

    print(
        f"[CAUCAFall] Epoch {epoch+1}/{FINETUNE_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )



🔹 Phase 2: CAUCAFall fine-tuning


KeyboardInterrupt: 

In [ ]:
torch.save(
    {
        "model_state": model.state_dict(),
        "clip_len": CLIP_LEN,
        "frame_size": FRAME_SIZE,
        "note": "Stage-1 3D CNN trained on CAUCAFall; saliency from conv3"
    },
    "stage1_3dcnn_saliency.pth"
)


# to extract weights from the model

# ckpt = torch.load("stage1_3dcnn_saliency.pth", map_location="cpu")

# model = Stage1_3DCNN(num_classes=2)
# model.load_state_dict(ckpt["model_state"])
# model.eval()

# with torch.no_grad():
#     logits, weights = model(clip, return_saliency=True)



In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
AVI_PATH = "/content/CAUCA_FALL/Subject.1/Fall forward/FallForwardS1.avi"   # <-- CHANGE THIS
CKPT_PATH = "/content/stage1_3dcnn_saliency.pth"

CLIP_LEN = 32         # must match training (e.g. 16 or 32)
FRAME_SIZE = 224       # must match training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- MODEL DEFINITION ----------------
class Stage1_3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.conv1 = nn.Conv3d(
            3, 64, kernel_size=(3,7,7),
            stride=(1,2,2), padding=(1,3,3)
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(
            kernel_size=(1,3,3),
            stride=(1,2,2), padding=(0,1,1)
        )

        self.conv2 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))

        self.conv3 = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(256)

        self.conv4 = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm3d(256)

        self.global_pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x, return_saliency=False):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))

        # ---- SALIENCY ----
        saliency = x.detach().pow(2).mean(dim=(1,3,4))
        saliency = saliency / (saliency.sum(dim=1, keepdim=True) + 1e-6)

        x = F.relu(self.bn4(self.conv4(x)))

        pooled = self.global_pool(x).flatten(1)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)

        if return_saliency:
            return logits, saliency
        return logits


# ---------------- LOAD MODEL ----------------
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model = Stage1_3DCNN(num_classes=2)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

# ---------------- LOAD AVI & EXTRACT FRAMES (UNIFORM SAMPLING) ----------------
cap = cv2.VideoCapture(AVI_PATH)
assert cap.isOpened(), f"Cannot open {AVI_PATH}"

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert total_frames > 0, "Empty video"

if total_frames >= CLIP_LEN:
    frame_indices = np.linspace(0, total_frames - 1, CLIP_LEN, dtype=int)
else:
    frame_indices = list(range(total_frames))
    frame_indices += [total_frames - 1] * (CLIP_LEN - total_frames)
    frame_indices = np.array(frame_indices, dtype=int)

frames = []

for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        raise RuntimeError(f"Failed to read frame {idx}")

    frame = cv2.resize(frame, (int(FRAME_SIZE), int(FRAME_SIZE)))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame)

cap.release()

# ---------------- BUILD CLIP TENSOR ----------------
frames_np = np.array(frames).astype(np.float32) / 255.0
clip = torch.from_numpy(frames_np).permute(3, 0, 1, 2).unsqueeze(0)
clip = clip.to(DEVICE)


# ---------------- RUN MODEL ----------------
with torch.no_grad():
    logits, weights = model(clip, return_saliency=True)

weights = weights.squeeze(0).cpu().numpy()

print("Logits:", logits.cpu().numpy())
print("Saliency weights:", weights)

# ---------------- VISUALIZATION ----------------
saliency_frame = np.repeat(weights, CLIP_LEN // len(weights))[:CLIP_LEN]

cols = 4
rows = int(np.ceil(CLIP_LEN / cols))

plt.figure(figsize=(cols * 4, rows * 4))
for i in range(CLIP_LEN):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(frames[i])
    plt.title(f"t={i} | w={saliency_frame[i]:.4f}")
    plt.axis("off")

plt.suptitle("Stage-1 Temporal Saliency (Frame-aligned)", fontsize=16)
plt.tight_layout()
plt.show()



Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
AVI_PATH = "/content/CAUCA_FALL/Subject.2/Fall backwards/FallBackwardsS2.avi"   # <-- CHANGE THIS
CKPT_PATH = "/content/stage1_3dcnn_saliency.pth"

CLIP_LEN = 32         # must match training (e.g. 16 or 32)
FRAME_SIZE = 224       # must match training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- MODEL DEFINITION ----------------
class Stage1_3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.conv1 = nn.Conv3d(
            3, 64, kernel_size=(3,7,7),
            stride=(1,2,2), padding=(1,3,3)
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(
            kernel_size=(1,3,3),
            stride=(1,2,2), padding=(0,1,1)
        )

        self.conv2 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))

        self.conv3 = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(256)

        self.conv4 = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm3d(256)

        self.global_pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x, return_saliency=False):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))

        # ---- SALIENCY ----
        saliency = x.detach().pow(2).mean(dim=(1,3,4))
        saliency = saliency / (saliency.sum(dim=1, keepdim=True) + 1e-6)

        x = F.relu(self.bn4(self.conv4(x)))

        pooled = self.global_pool(x).flatten(1)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)

        if return_saliency:
            return logits, saliency
        return logits


# ---------------- LOAD MODEL ----------------
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model = Stage1_3DCNN(num_classes=2)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

# ---------------- LOAD AVI & EXTRACT FRAMES (UNIFORM SAMPLING) ----------------
cap = cv2.VideoCapture(AVI_PATH)
assert cap.isOpened(), f"Cannot open {AVI_PATH}"

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert total_frames > 0, "Empty video"

if total_frames >= CLIP_LEN:
    frame_indices = np.linspace(0, total_frames - 1, CLIP_LEN, dtype=int)
else:
    frame_indices = list(range(total_frames))
    frame_indices += [total_frames - 1] * (CLIP_LEN - total_frames)
    frame_indices = np.array(frame_indices, dtype=int)

frames = []

for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        raise RuntimeError(f"Failed to read frame {idx}")

    frame = cv2.resize(frame, (int(FRAME_SIZE), int(FRAME_SIZE)))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame)

cap.release()

# ---------------- BUILD CLIP TENSOR ----------------
frames_np = np.array(frames).astype(np.float32) / 255.0
clip = torch.from_numpy(frames_np).permute(3, 0, 1, 2).unsqueeze(0)
clip = clip.to(DEVICE)


# ---------------- RUN MODEL ----------------
with torch.no_grad():
    logits, weights = model(clip, return_saliency=True)

weights = weights.squeeze(0).cpu().numpy()

print("Logits:", logits.cpu().numpy())
print("Saliency weights:", weights)

# ---------------- VISUALIZATION ----------------
saliency_frame = np.repeat(weights, CLIP_LEN // len(weights))[:CLIP_LEN]

cols = 4
rows = int(np.ceil(CLIP_LEN / cols))

plt.figure(figsize=(cols * 4, rows * 4))
for i in range(CLIP_LEN):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(frames[i])
    plt.title(f"t={i} | w={saliency_frame[i]:.4f}")
    plt.axis("off")

plt.suptitle("Stage-1 Temporal Saliency (Frame-aligned)", fontsize=16)
plt.tight_layout()
plt.show()



Output hidden; open in https://colab.research.google.com to view.